# 03 — Cross-subject results aggregation

Pulls all classification runs from `outputs/results/`, builds the cross-subject summary
table, and generates the plots that go into the GitHub README and any writeup.

**Models compared:** SVM (band-power features), EEGNet, CNN-BiLSTM
**Tasks:** N-class (per-subject), vowel/consonant, nasal/non-nasal, bilabial/non-bilabial

In [ ]:
import sys, json
sys.path.insert(0, '..')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

from src.evaluate import load_runs, plot_confusion_matrix, plot_per_subject_accuracy

df = load_runs('../outputs/results')
print(f'Total runs: {len(df)}')
print(df.head())

In [ ]:
# Aggregate per-task statistics
agg = df.groupby('task').agg(
    n_subjects=('subject', 'nunique'),
    mean_acc=('mean_accuracy', 'mean'),
    median_acc=('mean_accuracy', 'median'),
    std_acc=('mean_accuracy', 'std'),
    min_acc=('mean_accuracy', 'min'),
    max_acc=('mean_accuracy', 'max'),
    chance=('chance_level', 'first'),
).round(3).sort_values('mean_acc', ascending=False)
print(agg)

In [ ]:
# Per-subject summary across tasks (wide format)
wide = df.pivot_table(
    index='subject', columns='task',
    values='mean_accuracy', aggfunc='first'
).round(3)
print(wide.fillna('—'))

In [ ]:
# Plot: per-subject accuracy on the main classification task
for task in df['task'].unique():
    fig = plot_per_subject_accuracy(
        df, task=task,
        save_path=f'../outputs/figures/per_subject_{task.replace("-","_")}.png',
    )
    if fig is not None:
        plt.show()

In [ ]:
# Compare SVM vs EEGNet vs CNN-BiLSTM on the multi-class task
# (task names: '11-class' for SVM, '11-class_eegnet' for EEGNet, etc.)
model_results = []
for f in Path('../outputs/results').glob('**/*.json'):
    if f.name.startswith('_'):  # skip _batch_summary.json
        continue
    data = json.loads(f.read_text())
    task = data.get('task', '')
    # Heuristic: any task ending in _eegnet/_cnn_bilstm is a DL run
    if '_eegnet' in task:
        model = 'EEGNet'
    elif '_cnn_bilstm' in task or '_cnn-bilstm' in task:
        model = 'CNN-BiLSTM'
    elif task.endswith('-class'):
        model = 'SVM (band-power)'
    else:
        continue
    model_results.append({
        'subject': data['subject'],
        'model': model,
        'mean_acc': data['mean_accuracy'],
        'mean_bal': data.get('mean_balanced_accuracy'),
        'n_classes': data['n_classes'],
    })
model_df = pd.DataFrame(model_results)
if not model_df.empty:
    pivot = model_df.pivot_table(index='subject', columns='model', values='mean_acc', aggfunc='first').round(3)
    print(pivot.fillna('—'))

In [ ]:
# Confusion matrix for the best subject (any task)
if not df.empty:
    best = df.sort_values('mean_accuracy', ascending=False).iloc[0]
    print(f'Best run: {best["subject"]} on {best["task"]} = {best["mean_accuracy"]:.3f}')
    payload = json.loads(Path(best['file']).read_text())
    fig = plot_confusion_matrix(
        payload['confusion_matrix'], payload['classes'],
        title=f'{best["subject"]} — {best["task"]} (acc={best["mean_accuracy"]:.2f})',
        save_path=f'../outputs/figures/best_confusion.png',
    )
    plt.show()

In [ ]:
# Save the master summary CSV
csv_path = Path('../outputs/summary_results.csv')
df.to_csv(csv_path, index=False)
print(f'Saved {len(df)} rows to {csv_path}')